In [2]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

print(os.listdir("/content/drive/MyDrive")[:20])

['Classroom', "Prim's Algorithm.cpp", '2024IMG-030_DAA_A8_copy.pdf', '2024IMT-035_DAA_A8.pdf', 'Kavya Ravipati-oWZgLPV5Aqk_78aad5249facf87ecc75ce9b3ba2da35b04e31ab6bffd0bea0e33031_1772102823873.pdf', 'ID.pdf', 'Kavya hanishka ravipati resume.pdf', 'WhatsApp Image 2026-06-07 at 9.14.10 PM.jpeg', 'Screenshot 2026-06-07 211913.png', 'Screenshot 2026-06-07 212054.png', 'Colab Notebooks', 'KAVYA_resume.pdf', 'housing.csv', 'winequality-red.csv', 'AIML_Capstone_Kavya_Hanishka_Ravipati_2024IMT-035', 'VisDrone2019-DET-val.zip', 'V2.1']


In [4]:
import os

print(os.path.exists("/content/drive/MyDrive/V2.1"))
print(os.listdir("/content/drive/MyDrive/V2.1"))

True
['images', 'labels', 'data.yaml']


In [5]:
V21 = "/content/drive/MyDrive/V2.1"

TRAIN_IMG = os.path.join(V21, "images", "train")
TRAIN_LBL = os.path.join(V21, "labels", "train")

VAL_IMG = os.path.join(V21, "images", "val")
VAL_LBL = os.path.join(V21, "labels", "val")

TEST_IMG = os.path.join(V21, "images", "test")
TEST_LBL = os.path.join(V21, "labels", "test")

print("Train images:", os.path.exists(TRAIN_IMG))
print("Train labels:", os.path.exists(TRAIN_LBL))
print("Val images:", os.path.exists(VAL_IMG))
print("Test images:", os.path.exists(TEST_IMG))

Train images: True
Train labels: True
Val images: True
Test images: True


In [13]:
def get_images(folder):
    return [
        f for f in os.listdir(folder)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

train_images = get_images(TRAIN_IMG)
val_images = get_images(VAL_IMG)
test_images = get_images(TEST_IMG)

print("Train:", len(get_images(TRAIN_IMG)))
print("Val:", len(get_images(VAL_IMG)))
print("Test:", len(get_images(TEST_IMG)))

Train: 5078
Val: 1050
Test: 1084


In [8]:
!pip install -q ultralytics

from ultralytics import YOLO
import os
import torch

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Torch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


In [9]:
class_names = [
    "rov",
    "plant",
    "animal_fish",
    "animal_starfish",
    "animal_shells",
    "animal_crab",
    "animal_eel",
    "animal_etc",
    "trash_etc",
    "trash_fabric",
    "trash_fishing_gear",
    "trash_metal",
    "trash_paper",
    "trash_plastic",
    "trash_rubber",
    "trash_wood"
]

# Maximum number of copies for selected minority classes.
# 1 = original only
# 2 = original + 1 duplicate
# 3 = original + 2 duplicates

oversample_factor = {
    0: 1,   # rov
    1: 2,   # plant
    2: 1,   # animal_fish
    3: 2,   # animal_starfish
    4: 3,   # animal_shells
    5: 2,   # animal_crab
    6: 2,   # animal_eel
    7: 2,   # animal_etc
    8: 1,   # trash_etc
    9: 2,   # trash_fabric
    10: 2,  # trash_fishing_gear
    11: 1,  # trash_metal
    12: 3,  # trash_paper
    13: 1,  # trash_plastic
    14: 3,  # trash_rubber
    15: 2   # trash_wood
}

for i, name in enumerate(class_names):
    print(f"{i:2d} {name:20s} -> {oversample_factor[i]}x")

 0 rov                  -> 1x
 1 plant                -> 2x
 2 animal_fish          -> 1x
 3 animal_starfish      -> 2x
 4 animal_shells        -> 3x
 5 animal_crab          -> 2x
 6 animal_eel           -> 2x
 7 animal_etc           -> 2x
 8 trash_etc            -> 1x
 9 trash_fabric         -> 2x
10 trash_fishing_gear   -> 2x
11 trash_metal          -> 1x
12 trash_paper          -> 3x
13 trash_plastic        -> 1x
14 trash_rubber         -> 3x
15 trash_wood           -> 2x


In [14]:
from collections import Counter
import os

manifest = []
factor_distribution = Counter()

for img in train_images:

    label_name = os.path.splitext(img)[0] + ".txt"
    label_path = os.path.join(TRAIN_LBL, label_name)

    classes = []

    if os.path.exists(label_path):
        with open(label_path, "r") as f:
            for line in f:
                parts = line.strip().split()

                if len(parts) >= 5:
                    classes.append(int(parts[0]))

    # If image contains multiple classes,
    # use the highest oversampling factor
    factor = max(
        [oversample_factor.get(c, 1) for c in classes],
        default=1
    )

    img_path = os.path.join(TRAIN_IMG, img)

    for _ in range(factor):
        manifest.append(img_path)

    factor_distribution[factor] += 1

print("Original train images:", len(train_images))
print("Oversampled train entries:", len(manifest))
print("Additional entries:", len(manifest) - len(train_images))

print("\nImage factor distribution:")
for k, v in sorted(factor_distribution.items()):
    print(f"{k}x : {v} images")

Original train images: 5078
Oversampled train entries: 7134
Additional entries: 2056

Image factor distribution:
1x : 3364 images
2x : 1372 images
3x : 342 images


In [17]:
WORK_DIR = "/content/V2.1_oversampled"
os.makedirs(WORK_DIR, exist_ok=True)

TRAIN_LIST = os.path.join(
    WORK_DIR,
    "train_oversampled.txt"
)

with open(TRAIN_LIST, "w") as f:
    for path in manifest:
        f.write(path + "\n")

print("Saved:", TRAIN_LIST)
print("Total entries:", len(manifest))

Saved: /content/V2.1_oversampled/train_oversampled.txt
Total entries: 7134


In [16]:
with open(TRAIN_LIST, "r") as f:
    entries = [x.strip() for x in f if x.strip()]

print("Total training entries:", len(entries))
print("Unique training images:", len(set(entries)))
print("Repeated entries:", len(entries) - len(set(entries)))

assert len(set(entries)) == len(train_images)

print("\nOversampling verification passed ✓")

Total training entries: 7134
Unique training images: 5078
Repeated entries: 2056

Oversampling verification passed ✓


In [18]:
import yaml

DATA_YAML = os.path.join(WORK_DIR, "data.yaml")

data = {
    "train": TRAIN_LIST,
    "val": VAL_IMG,
    "test": TEST_IMG,
    "nc": 16,
    "names": class_names
}

with open(DATA_YAML, "w") as f:
    yaml.dump(data, f, sort_keys=False)

print(open(DATA_YAML).read())

train: /content/V2.1_oversampled/train_oversampled.txt
val: /content/drive/MyDrive/V2.1/images/val
test: /content/drive/MyDrive/V2.1/images/test
nc: 16
names:
- rov
- plant
- animal_fish
- animal_starfish
- animal_shells
- animal_crab
- animal_eel
- animal_etc
- trash_etc
- trash_fabric
- trash_fishing_gear
- trash_metal
- trash_paper
- trash_plastic
- trash_rubber
- trash_wood



In [19]:
print("Train unique images:", len(train_images))
print("Val images:", len(val_images))
print("Test images:", len(test_images))

print("\nTrain labels:", len(os.listdir(TRAIN_LBL)))
print("Val labels:", len(os.listdir(VAL_LBL)))
print("Test labels:", len(os.listdir(TEST_LBL)))

Train unique images: 5078
Val images: 1050
Test images: 1084

Train labels: 5078
Val labels: 1050
Test labels: 1084


In [ ]:
with open(manifest, "r") as f:
    entries = [x.strip() for x in f if x.strip()]

print("Unique training images:", len(set(entries)))
print("Total training entries:", len(entries))

val_imgs = os.listdir(os.path.join(V21, "images", "val"))
test_imgs = os.listdir(os.path.join(V21, "images", "test"))

val_imgs = [x for x in val_imgs if x.lower().endswith((".jpg",".jpeg",".png"))]
test_imgs = [x for x in test_imgs if x.lower().endswith((".jpg",".jpeg",".png"))]

print("Validation images:", len(val_imgs))
print("Test images:", len(test_imgs))

Unique training images: 5078
Total training entries: 6703
Validation images: 1050
Test images: 1084


In [21]:
from ultralytics import YOLO

model = YOLO("yolo11s.pt")

results = model.train(
    data=DATA_YAML,
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,
    workers=4,
    pretrained=True,
    seed=42,
    project="/content/drive/MyDrive/material_version/yolo_runs",
    name="V2.1_yolo11s_oversampled_640"
)

Ultralytics 8.4.173 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/V2.1_oversampled/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=V2.1_yolo11s_oversampled_640, nbs=64, nms=None, opse